# EA1 - Carga y Reporte de Datos Arriendo por estrato colombia 2026

# Integrantes

Natalia Jadith Castro Ospino

Oscar Luis Marquez Arrieta

# Preparar e Importar la clase de carga

In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases.hf_loader import HFDatasetLoader

# Configurar DataSet

In [2]:
loader = HFDatasetLoader(
    dataset_name="aptologia/arriendo-por-estrato-colombia-2026",
    split="train",
    filename="datos/arriendo-por-estrato-colombia-2026.csv",
)

# Descargar Dataset y crear DataFrame

In [3]:
df = loader.load()
df.head(3)

README.md:   0%|          | 0.00/4.75k [00:00<?, ?B/s]

precio_por_estrato.csv:   0%|          | 0.00/1.66k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/32 [00:00<?, ? examples/s]

,fecha,ciudad_id,ciudad,estrato,avisos,mediana,por_m2
0,2026-09-03,barranquilla,Barranquilla,3,135,1500000,25747
1,2026-09-03,barranquilla,Barranquilla,4,419,2600000,33505
2,2026-09-03,barranquilla,Barranquilla,5,461,3050000,36706


# Estructura de DataFrame

In [4]:
print(f"Filas: {len(df)}")
print(f"Columnas: {len(df.columns)}")
df.info()

Filas: 32
Columnas: 7
<class 'pandas.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   fecha      32 non-null     str  
 1   ciudad_id  32 non-null     str  
 2   ciudad     32 non-null     str  
 3   estrato    32 non-null     int64
 4   avisos     32 non-null     int64
 5   mediana    32 non-null     int64
 6   por_m2     32 non-null     int64
dtypes: int64(4), str(3)
memory usage: 2.7 KB


# Importacion de clase reportes

In [5]:
from clases.reportes import Reportes

# visualizar reportes como texto

In [6]:
reportes = Reportes()
texto = reportes.generar_reporte(df)

print(texto)

REPORTE DEL DATAFRAME
Filas: 32
Columnas: 7
Duplicados: 0

Columnas y tipos:
- fecha: str
- ciudad_id: str
- ciudad: str
- estrato: int64
- avisos: int64
- mediana: int64
- por_m2: int64

Valores nulos por columna:
- fecha: 0
- ciudad_id: 0
- ciudad: 0
- estrato: 0
- avisos: 0
- mediana: 0
- por_m2: 0

Estadísticas numéricas:
       estrato     avisos       mediana     por_m2
count   32.000     32.000  3.200000e+01     32.000
mean     4.188   4781.250  3.425906e+06  34175.875
std      1.355   7912.354  2.504060e+06  10759.204
min      2.000    135.000  7.500000e+05  13000.000
25%      3.000    574.250  1.587500e+06  28373.250
50%      4.000   1239.000  2.800000e+06  33419.000
75%      5.000   4836.000  4.562500e+06  39347.500
max      6.000  31084.000  1.100000e+07  59524.000



# Imprimir Reportes

In [7]:
reportes.imprimir_reporte(df)

REPORTE DEL DATAFRAME
Filas: 32
Columnas: 7
Duplicados: 0

Columnas y tipos:
- fecha: str
- ciudad_id: str
- ciudad: str
- estrato: int64
- avisos: int64
- mediana: int64
- por_m2: int64

Valores nulos por columna:
- fecha: 0
- ciudad_id: 0
- ciudad: 0
- estrato: 0
- avisos: 0
- mediana: 0
- por_m2: 0

Estadísticas numéricas:
       estrato     avisos       mediana     por_m2
count   32.000     32.000  3.200000e+01     32.000
mean     4.188   4781.250  3.425906e+06  34175.875
std      1.355   7912.354  2.504060e+06  10759.204
min      2.000    135.000  7.500000e+05  13000.000
25%      3.000    574.250  1.587500e+06  28373.250
50%      4.000   1239.000  2.800000e+06  33419.000
75%      5.000   4836.000  4.562500e+06  39347.500
max      6.000  31084.000  1.100000e+07  59524.000


# Guardar Reporte en archivo texto

In [8]:
# Guardar el reporte del DataFrame original
ruta_original = (
    ROOT / "Sincronizar" / "reportes" /
    "arriendo-por-estrato-colombia-2026_original.txt"
)

reportes.guardar_reporte(df, ruta_original)

print(f"Reporte original guardado en: {ruta_original}")

Reporte original guardado en: /home/runner/work/Projects_Machine_Learning2026/Projects_Machine_Learning2026/Sincronizar/reportes/arriendo-por-estrato-colombia-2026_original.txt


# Transformaciones del DataFrame

Realizamos cuatro transformaciones clave al conjunto de datos sobre arriendos por estrato en Colombia. Estas incluyen la conversión de fechas, la selección de columnas relevantes, el filtrado según el estrato y la creación de una nueva variable que nos permitirá expresar el arriendo mediano en millones de pesos.

In [9]:
# Copia el DataFrame original para conservar los datos cargados
df_transformado = df.copy()

#Transformación 1: Convertir la columna fecha al tipo datetime
df_transformado["fecha"] = pd.to_datetime(df_transformado["fecha"])
print("Transformaciones 1: Conversión de fecha")
print(df_transformado[["fecha"]].head())
print("Tipo de dato:", df_transformado["fecha"].dtype)

# Transformación 2: Seleccionar las columnas relevantes
columnas = [
    "fecha", "ciudad", "estrato",
    "avisos", "mediana", "por_m2"
]
df_transformado = df_transformado[columnas]
print("\nTransformación 2: Selección de columnas")
display(df_transformado.head())

# Transformación 3: Filtrar los registros de estratos 3 a 6
df_transformado = df_transformado[
    df_transformado["estrato"].between(3, 6) 
].copy()
print("\nTransformación 3: Filtro de estrato 3, 6")
print("Filas después del filtro:", len(df_transformado))
display(df_transformado.head())

# Transformación 4: Crear una variable derivada en millones de pesos
df_transformado["mediana_millones"] = (
    df_transformado["mediana"] / 1_000_000
).round(2)
print("\nTransformación 4: arriendo mediano en millones")
display(
    df_transformado[
        ["ciudad", "estrato", "mediana", "mediana_millones"]
    ].head()
)


Transformaciones 1: Conversión de fecha
       fecha
0 2026-09-03
1 2026-09-03
2 2026-09-03
3 2026-09-03
4 2026-09-03
Tipo de dato: datetime64[us]

Transformación 2: Selección de columnas


,fecha,ciudad,estrato,avisos,mediana,por_m2
0,2026-09-03,Barranquilla,3,135,1500000,25747
1,2026-09-03,Barranquilla,4,419,2600000,33505
2,2026-09-03,Barranquilla,5,461,3050000,36706
3,2026-09-03,Barranquilla,6,378,4550000,38889
4,2026-09-03,Bogotá,2,1117,1200000,25000



Transformación 3: Filtro de estrato 3, 6
Filas después del filtro: 28


,fecha,ciudad,estrato,avisos,mediana,por_m2
0,2026-09-03,Barranquilla,3,135,1500000,25747
1,2026-09-03,Barranquilla,4,419,2600000,33505
2,2026-09-03,Barranquilla,5,461,3050000,36706
3,2026-09-03,Barranquilla,6,378,4550000,38889
5,2026-09-03,Bogotá,3,7543,1600000,36111



Transformación 4: arriendo mediano en millones


,ciudad,estrato,mediana,mediana_millones
0,Barranquilla,3,1500000,1.50
1,Barranquilla,4,2600000,2.60
2,Barranquilla,5,3050000,3.05
3,Barranquilla,6,4550000,4.55
5,Bogotá,3,1600000,1.60


In [10]:
# Genera el reporte final del DataFrame transformado
reportes = Reportes()

texto_transformado = reportes.generar_reporte(df_transformado)
print(texto_transformado)

# Guarda el reporte en la carpeta de reportes del proyecto
ruta_reporte = (
    ROOT / "Sincronizar" / "reportes" /
    "arriendo-por-estrato-colombia-2026.txt"
)

reportes.guardar_reporte(df_transformado, str(ruta_reporte))

print(f"\nReporte transformado guardado en: {ruta_reporte}")


REPORTE DEL DATAFRAME
Filas: 28
Columnas: 7
Duplicados: 0

Columnas y tipos:
- fecha: datetime64[us]
- ciudad: str
- estrato: int64
- avisos: int64
- mediana: int64
- por_m2: int64
- mediana_millones: float64

Valores nulos por columna:
- fecha: 0
- ciudad: 0
- estrato: 0
- avisos: 0
- mediana: 0
- por_m2: 0
- mediana_millones: 0

Estadísticas numéricas:
       estrato     avisos       mediana     por_m2  mediana_millones
count   28.000     28.000  2.800000e+01     28.000            28.000
mean     4.500   5358.250  3.752821e+06  36008.500             3.753
std      1.139   8311.462  2.508602e+06   9928.026             2.509
min      3.000    135.000  1.050000e+06  18000.000             1.050
25%      3.750    687.750  2.037500e+06  30430.500             2.038
50%      4.500   1440.500  3.025000e+06  34169.500             3.025
75%      5.250   5704.000  4.651500e+06  40333.250             4.652
max      6.000  31084.000  1.100000e+07  59524.000            11.000


Reporte transformado